# INF2003 Database Systems Project - Movie Review and Discovery System

<h2 style="color: #17365D;">movies.csv Dataset Preparation</h2>

<h2 style="color: #580018;">Data Understanding</h2>

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv("imdb_top_1000_movies.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
# Check the number of rows and columns in the dataset
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 1000
Number of columns: 16


In [4]:
# Display the original column names
df.columns.tolist()

['Poster_Link',
 'Series_Title',
 'Released_Year',
 'Certificate',
 'Runtime',
 'Genre',
 'IMDB_Rating',
 'Overview',
 'Meta_score',
 'Director',
 'Star1',
 'Star2',
 'Star3',
 'Star4',
 'No_of_Votes',
 'Gross']

<h2 style="color: #580018;">Data Transformation</h2>

In [5]:
# Generate unique movie IDs
df["movie_id"] = [
    f"M{i:04d}" for i in range(1, len(df) + 1)
]

In [6]:
# Combine the four star columns into one cast column
star_columns = ["Star1", "Star2", "Star3", "Star4"]

df["cast"] = df[star_columns].fillna("").apply(
    lambda row: ", ".join(
        name.strip() for name in row if name.strip()
    ),
    axis=1
)

In [7]:
# Remove unwanted columns
df.drop(
    columns=["IMDB_Rating", "Meta_score", "Gross", "Star1", "Star2", "Star3", "Star4"],
    inplace=True
)

print("Remaining columns:", df.shape[1])

Remaining columns: 11


In [8]:
# Rename columns to match the final data dictionary
df.rename(columns={
    "Poster_Link": "poster_url",
    "Series_Title": "title",
    "Released_Year": "release_year",
    "Certificate": "certificate",
    "Runtime": "duration",
    "Genre": "genre",
    "Overview": "description",
    "Director": "director",
    "No_of_Votes": "imdb_votes"
}, inplace=True)

In [9]:
# Arrange columns in the final required order
final_column_order = [
    "movie_id",
    "poster_url",
    "title",
    "release_year",
    "certificate",
    "duration",
    "genre",
    "description",
    "director",
    "cast",
    "imdb_votes"
]

df = df[final_column_order]

In [10]:
df.head() #preview of the dataset

,movie_id,poster_url,title,release_year,certificate,duration,genre,description,director,cast,imdb_votes
0,M0001,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,Two imprisoned men bond over a number of years...,Frank Darabont,"Tim Robbins, Morgan Freeman, Bob Gunton, Willi...",2343110
1,M0002,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",An organized crime dynasty's aging patriarch t...,Francis Ford Coppola,"Marlon Brando, Al Pacino, James Caan, Diane Ke...",1620367
2,M0003,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...,Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart, M...",2303232
3,M0004,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",The early life and career of Vito Corleone in ...,Francis Ford Coppola,"Al Pacino, Robert De Niro, Robert Duvall, Dian...",1129952
4,M0005,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",A jury holdout attempts to prevent a miscarria...,Sidney Lumet,"Henry Fonda, Lee J. Cobb, Martin Balsam, John ...",689845


<h2 style="color: #580018;">Data Cleaning and Preprocessing</h2>

In [11]:
# Check for duplicated rows
print("Duplicated rows:", df.duplicated().sum())

Duplicated rows: 0


In [12]:
# Remove whitespace from text fields
text_columns = [
    "poster_url", "title", "release_year", "certificate",
    "duration", "genre", "description", "director", "cast"
]

for column in text_columns:
    df[column] = df[column].str.strip()

In [13]:
# Remove " min" from duration
df["duration"] = df["duration"].str.replace(
    " min", "", regex=False
)

In [14]:
# Convert numeric columns to appropriate data types
df["release_year"] = pd.to_numeric(
    df["release_year"], errors="coerce"
).astype("Int64")

df["duration"] = pd.to_numeric(
    df["duration"], errors="coerce"
).astype("Int64")

df["imdb_votes"] = pd.to_numeric(
    df["imdb_votes"], errors="coerce"
).astype("Int64")

In [15]:
# Check for invalid numeric values
print("Invalid numeric values:")
print((df[["release_year", "duration", "imdb_votes"]] <= 0).sum())

Invalid numeric values:
release_year    0
duration        0
imdb_votes      0
dtype: Int64


In [16]:
df.info() # Basic data understanding

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   movie_id      1000 non-null   object
 1   poster_url    1000 non-null   object
 2   title         1000 non-null   object
 3   release_year  999 non-null    Int64 
 4   certificate   899 non-null    object
 5   duration      1000 non-null   Int64 
 6   genre         1000 non-null   object
 7   description   1000 non-null   object
 8   director      1000 non-null   object
 9   cast          1000 non-null   object
 10  imdb_votes    1000 non-null   Int64 
dtypes: Int64(3), object(8)
memory usage: 89.0+ KB


In [17]:
# Checking for missing values
missing_values = df.isna().sum()
missing_values

movie_id          0
poster_url        0
title             0
release_year      1
certificate     101
duration          0
genre             0
description       0
director          0
cast              0
imdb_votes        0
dtype: int64

In [18]:
# Find the movie with the missing release year
display(
    df[df["release_year"].isna()][
        ["movie_id", "title", "release_year"]
    ]
)

,movie_id,title,release_year
966,M0967,Apollo 13,<NA>


In [19]:
# Correct the release year for Apollo 13 (based on research)
df.loc[df["title"] == "Apollo 13", "release_year"] = 1995

In [20]:
# Replace missing and blank certificates
df["certificate"] = df["certificate"].replace("", pd.NA)
df["certificate"] = df["certificate"].fillna("Unknown")

In [21]:
# Check missing values after type conversion and cleaning
missing_values = df.isna().sum()
missing_values

movie_id        0
poster_url      0
title           0
release_year    0
certificate     0
duration        0
genre           0
description     0
director        0
cast            0
imdb_votes      0
dtype: int64

<h2 style="color: #580018;">Data Preview and Export</h2>

In [22]:
df.head() # Preview of the cleaned movies dataset

,movie_id,poster_url,title,release_year,certificate,duration,genre,description,director,cast,imdb_votes
0,M0001,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142,Drama,Two imprisoned men bond over a number of years...,Frank Darabont,"Tim Robbins, Morgan Freeman, Bob Gunton, Willi...",2343110
1,M0002,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175,"Crime, Drama",An organized crime dynasty's aging patriarch t...,Francis Ford Coppola,"Marlon Brando, Al Pacino, James Caan, Diane Ke...",1620367
2,M0003,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...,Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart, M...",2303232
3,M0004,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202,"Crime, Drama",The early life and career of Vito Corleone in ...,Francis Ford Coppola,"Al Pacino, Robert De Niro, Robert Duvall, Dian...",1129952
4,M0005,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96,"Crime, Drama",A jury holdout attempts to prevent a miscarria...,Sidney Lumet,"Henry Fonda, Lee J. Cobb, Martin Balsam, John ...",689845


In [23]:
df.to_csv("movies.csv", index=False) # Export the cleaned movies dataset

<h2 style="color: #17365D;">genres.csv Dataset Preparation</h2>

In [24]:
# Extract individual genres from the comma-separated genre column
genre_list = df["genre"].dropna().str.split(", ").explode()

# Remove whitespace and obtain unique genre names
unique_genres = sorted(
    genre.strip()
    for genre in genre_list.unique()
    if genre.strip()
)

# Preview the unique genres
print("Number of unique genres:", len(unique_genres))
print(unique_genres)

Number of unique genres: 21
['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sport', 'Thriller', 'War', 'Western']


In [25]:
# Create a genres table with unique IDs
genres_df = pd.DataFrame({
    "genre_id": [
        f"G{i:03d}" for i in range(1, len(unique_genres) + 1)
    ],
    "genre_name": unique_genres
})

In [26]:
genres_df.head() # Preview the cleaned genres dataset

,genre_id,genre_name
0,G001,Action
1,G002,Adventure
2,G003,Animation
3,G004,Biography
4,G005,Comedy


In [27]:
genres_df.to_csv("genres.csv", index=False) # Export the cleaned genres dataset

<h2 style="color: #17365D;">movie_genres.csv Dataset Preparation</h2>

In [28]:
# Split each movie's genres into individual movie-genre relationships
movie_genres_df = df[["movie_id", "genre"]].copy()

movie_genres_df["genre_name"] = (
    movie_genres_df["genre"].str.split(", ")
)

movie_genres_df = movie_genres_df.explode("genre_name")

In [29]:
# Remove whitespace from genre names
movie_genres_df["genre_name"] = (
    movie_genres_df["genre_name"].str.strip()
)

In [30]:
# Match each genre name to its corresponding genre ID
movie_genres_df = movie_genres_df.merge(
    genres_df,
    on="genre_name",
    how="left"
)

# Keep only the columns required for the relationship table
movie_genres_df = movie_genres_df[
    ["movie_id", "genre_id"]
]

In [31]:
# Preview the movie-genres dataset
movie_genres_df.head()

,movie_id,genre_id
0,M0001,G007
1,M0002,G006
2,M0002,G007
3,M0003,G001
4,M0003,G006


In [32]:
# Export the movie-genre relationship table
movie_genres_df.to_csv("movie_genres.csv", index=False)

In [1]:
# Check the actual filenames in the current folder
import os

for file in os.listdir():
    if file.endswith((".csv", ".xls", ".xlsx")):
        print(file)

genres.csv
imdb_top_1000_movies.csv
movies.csv
movie_genres.csv
